# 01 · Analysing the randomised experiment
**Project:** *Who should we target?* Causal uplift modelling on the Criteo randomised ad experiment.
**Data:** [Criteo Uplift Prediction Dataset](https://ailab.criteo.com/criteo-uplift-prediction-dataset/) (Diemert et al., 2021),
13.9M users. 85% were eligible to see ads (treatment) and 15% were held out (control). Outcomes: site **visit** and **conversion**.
We work with a stratified 10% sample (about 1.4M users), downloaded automatically from Hugging Face (`criteo/criteo-uplift`)
or Kaggle (`arashnic/uplift-modeling`, which uses your `KAGGLE_API_TOKEN` Colab secret). **Runtime: CPU is enough**

In [ ]:
# ---------- Setup: run this cell first (CPU runtime is enough) ----------
import os, sys, subprocess, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("UP_SMOKE") == "1"          # local smoke test on synthetic data
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if SMOKE:
    PROJECT_DIR = os.environ["UP_PROJECT_DIR"]
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = sorted(glob.glob("/content/drive/MyDrive/**/uplift/src/uplift/data.py", recursive=True), key=len)
    assert hits, "Upload the 'uplift' folder to your Google Drive first."
    PROJECT_DIR = str(Path(hits[0]).parents[2])                     # found automatically, wherever you put it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)
else:   # local Jupyter / VS Code: notebook lives in uplift/notebooks/
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART, FIG = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures"
DATA = "/content/data" if ON_COLAB else f"{PROJECT_DIR}/data"
for d in (ART, FIG, DATA): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from uplift.data import FEATURES, load_criteo, make_synthetic
from uplift import experiment as E, learners as L, metrics as M, causal as C
SEED = 42
df = make_synthetic(60_000) if SMOKE else load_criteo(DATA, sample_frac=0.10)   # 10% sample ~ 1.4M users
TREAT_SHARE = 0.85                                                              # by design of the experiment
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=float)
def load_json(name):
    return json.load(open(f"{ART}/{name}.json"))
print(f"{len(df):,} users | treated share {df.treatment.mean():.3f} | visit {df.visit.mean():.4f} | conversion {df.conversion.mean():.5f}")

## 1. Sample-ratio mismatch and covariate balance 
Healthy randomization check

In [ ]:
srm = E.srm_test(df.treatment, TREAT_SHARE); print(srm)
bal = E.balance_table(df, FEATURES)
print(f"max |SMD| = {bal.smd.abs().max():.4f}  (< 0.1 = balanced)")
fig, ax = plt.subplots(figsize=(6, 3.5)); bal.smd.plot.bar(ax=ax, color="#4C72B0")
ax.axhline(0.1, ls="--", c="r", lw=1); ax.axhline(-0.1, ls="--", c="r", lw=1); ax.set_title("Covariate balance (standardised mean difference)")
savefig("01_balance"); bal.round(4)

## 2. Average treatment effect 
does advertising work on average?

In [ ]:
ate = {y: E.ate_binary(df[y], df.treatment) for y in ["visit", "conversion"]}
ate_df = pd.DataFrame(ate).T
ate_df[["rate_treated", "rate_control", "ate", "ci_low", "ci_high", "relative_lift", "p_value"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, y in zip(ax, ["visit", "conversion"]):
    r = ate[y]
    a.bar(["control", "treated"], [r["rate_control"], r["rate_treated"]], color=["#BBBBBB", "#C44E52"])
    a.set_title(f"{y}: +{r['ate']*100:.3f} pp ({r['relative_lift']:.0%} relative)")
savefig("01_ate")

## 3. Data for the test
How small an effect can this sample detect? 
How many users would a conversion test need?

In [ ]:
n1, n0 = int(df.treatment.sum()), int((1 - df.treatment).sum())
power = []
for y in ["visit", "conversion"]:
    p0 = ate[y]["rate_control"]
    power.append({"outcome": y, "control_rate": p0, "observed_ate": ate[y]["ate"],
                  "MDE (80% power)": E.mde(p0, n1, n0),
                  "control users needed for observed effect": E.required_n_per_group(p0, ate[y]["ate"], ratio=n1 / n0)})
power = pd.DataFrame(power).set_index("outcome"); power

## 4. Heterogeneity
is the effect the same for everyone?

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey=True)
het = {}
for ax, f in zip(axes.ravel(), FEATURES):
    h = E.effect_by_quantile(df, f, "visit", q=5); het[f] = h       # value-based bins (features have many ties)
    if len(h):
        ax.errorbar(range(len(h)), h.ate, yerr=[h.ate - h.ci_low, h.ci_high - h.ate], fmt="o-", capsize=3)
        ax.set_xticks(range(len(h)), h.index)
    ax.axhline(ate["visit"]["ate"], c="r", ls="--", lw=1); ax.set_title(f"{f} ({len(h)} bins)")
fig.suptitle("Visit uplift by feature quintile (red = overall ATE; fewer bins = heavily tied feature)"); savefig("01_heterogeneity")
spread = pd.Series({f: (h.ate.max() - h.ate.min()) if len(h) > 1 else 0.0 for f, h in het.items()}).sort_values(ascending=False)
spread.to_frame("max - min bin ATE").head(6)

In [ ]:
save_json({"srm": srm, "max_abs_smd": float(bal.smd.abs().max()), "ate": ate, "power": power.reset_index().to_dict(orient="records"),
           "n": len(df), "heterogeneity_spread_top": spread.head(5).to_dict()}, "experiment")
print("saved")